In [ ]:
!cp -r /content/drive/MyDrive/dataset /content/
!mkdir -p /content/dataset
!tar -xf /content/dataset/dataset.tar -C /content/dataset

In [ ]:
import tensorflow as tf
import numpy as np

# cargar modelo sin compilar
model = tf.keras.models.load_model(
    '/content/drive/MyDrive/model/model_epoch_063.keras',
    compile=False
)

# convertidor
converter = tf.lite.TFLiteConverter.from_keras_model(model)

# optimización
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# cuantización float16
converter.target_spec.supported_types = [tf.float16]

print("Convirtiendo modelo...")

tflite_model = converter.convert()

output_path = '/content/drive/MyDrive/models/lunar_rpi5_float16.tflite'

with open(output_path, 'wb') as f:
    f.write(tflite_model)

print("Modelo convertido")
print("Tamaño:", len(tflite_model)/(1024*1024), "MB")

Convirtiendo modelo...
Saved artifact at '/tmp/tmpiekh6366'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 240, 320, 3), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 240, 320, 5), dtype=tf.float32, name=None)
Captures:
  134034152784144: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134034152783952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134034152788560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134034152788368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134034152787600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134034152789712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134034152789904: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134034152786448: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134034152790480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134034152788944: TensorSpec(shape=(), dtype=tf.resou

In [ ]:
import tensorflow as tf
import numpy as np
import cv2
import os

MODEL_PATH = "/content/drive/MyDrive/model/model_epoch_063.keras"
OUTPUT_PATH = "/content/drive/MyDrive/models/lunar_rpi5_int8.tflite"

IMG_H = 240
IMG_W = 320

# cargar modelo
model = tf.keras.models.load_model(MODEL_PATH, compile=False)

# ==============================
# dataset representativo
# ==============================

DATASET_DIR = "/content/dataset/color"

image_paths = [
    os.path.join(DATASET_DIR,x)
    for x in os.listdir(DATASET_DIR)
    if x.endswith(".jpg") or x.endswith(".png")
][:500]   # 100-500 imágenes es suficiente

def representative_dataset():

    for path in image_paths:

        img = cv2.imread(path)
        img = cv2.resize(img,(IMG_W,IMG_H))

        img = img.astype(np.float32) / 255.0
        img = np.expand_dims(img,0)

        yield [img]

# ==============================
# conversión
# ==============================

converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = [tf.lite.Optimize.DEFAULT]

converter.representative_dataset = representative_dataset

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

converter.inference_input_type = tf.uint8
converter.inference_output_type = tf.uint8

print("Convirtiendo modelo a INT8...")

tflite_model = converter.convert()

with open(OUTPUT_PATH,"wb") as f:
    f.write(tflite_model)

print("Modelo exportado:", OUTPUT_PATH)
print("Tamaño:", len(tflite_model)/(1024*1024), "MB")

Convirtiendo modelo a INT8...
Saved artifact at '/tmp/tmpk3f7cpxw'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 240, 320, 3), dtype=tf.float32, name='input_layer')
Output Type:
  TensorSpec(shape=(None, 240, 320, 5), dtype=tf.float32, name=None)
Captures:
  137314268519824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137314268521360: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137314268520592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137314268521744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137314268521552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137314268512336: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137314268522896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137314268523664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137314268522512: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137314268524048: TensorSpec(shape=(), dtype=t

/usr/local/lib/python3.11/dist-packages/tensorflow/lite/python/convert.py:997: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Modelo exportado: /content/drive/MyDrive/models/lunar_rpi5_int8.tflite
Tamaño: 1.0982513427734375 MB
